# Modul 3: Optimizer dan Strategi Pelatihan

**Nama:** ISI NAMA  
**NIM:** ISI NIM  
**Kelas:** ISI KELAS  
**Tanggal:** YYYY-MM-DD  

Simpan berkas ini sebagai `M03_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Protokol modul tetap: subset $12\,000$/$3\,000$, model $784 \rightarrow 128 \rightarrow 10$, batch $128$, $5$ epoch. Ubah **hanya** faktor yang sedang diuji.
3. Seluruh run wajib memanggil fungsi `jalankan()` yang sama.
4. Pilih model dari validation loss. Data uji dipakai **satu kali** di akhir.
5. Catat setiap run ke `metrics.csv`, termasuk run yang gagal.
6. Luaran: `M03_NIM.ipynb`, `M03_NIM.pdf`, `M03_NIM_metrics.csv`, dan grafik perbandingan.

In [ ]:
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.datasets import FashionMNIST

NIM = 'TODO'                     # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE), 'seed': SEED})

## A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum dimulai.

1. **Jumlah update satu epoch bila $n=12\,000$ dan batch $=128$:** TODO
2. **Apa itu derau gradien pada mini-batch, dan mengapa full-batch tidak memilikinya:** TODO
3. **Mengapa membandingkan dua optimizer pada learning rate yang sama belum tentu adil:** TODO
4. **Perkiraan bentuk kurva loss bila learning rate dinaikkan sepuluh kali:** TODO

**Hipotesis awal.** Optimizer mana yang Anda perkirakan menang, dan atas dasar apa? TODO

## B. Data dan protokol - 15 poin

Ambil subset terstratifikasi dan hitung statistik normalisasi **hanya** dari subset latih.

In [ ]:
DATA_ROOT = Path('../../data/raw')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data/raw')

latih_penuh = FashionMNIST(root=DATA_ROOT, train=True, download=False,
                           transform=transforms.ToTensor())
uji_resmi = FashionMNIST(root=DATA_ROOT, train=False, download=False,
                         transform=transforms.ToTensor())

X_penuh = latih_penuh.data.float().unsqueeze(1) / 255.0
y_penuh = latih_penuh.targets
X_uji = uji_resmi.data.float().unsqueeze(1) / 255.0
y_uji = uji_resmi.targets

# TODO 1: ambil 12.000 latih dan 3.000 validasi secara terstratifikasi
#         memakai train_test_split dengan random_state=SEED.
idx_latih, idx_val = ...

X_latih, y_latih = X_penuh[idx_latih], y_penuh[idx_latih]
X_val, y_val = X_penuh[idx_val], y_penuh[idx_val]

# TODO 2: hitung MEAN dan STD dari subset LATIH saja, lalu normalkan ketiga split.
MEAN, STD = ..., ...
normalkan = lambda t: (t - MEAN) / STD

ds_latih = TensorDataset(normalkan(X_latih), y_latih)
ds_val = TensorDataset(normalkan(X_val), y_val)
ds_uji = TensorDataset(normalkan(X_uji), y_uji)

print(f'latih {len(ds_latih)}  validasi {len(ds_val)}  uji {len(ds_uji)}')
print('distribusi kelas latih:', torch.bincount(y_latih).tolist())

# Pemeriksaan wajib.
assert (len(ds_latih), len(ds_val), len(ds_uji)) == (12_000, 3_000, 10_000)
assert torch.bincount(y_latih).min().item() == 1_200, 'subset harus terstratifikasi'
print('split sesuai protokol')

In [ ]:
BATCH = 128
EPOCH = 5

def buat_model():
    """TODO 3: kembalikan model 784 -> 128 -> 10 yang SELALU dimulai dari seed sama."""
    raise NotImplementedError

def buat_optimizer(nama, params, lr):
    """TODO 4: kembalikan SGD, SGD+momentum 0.9, atau Adam sesuai nama."""
    raise NotImplementedError

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds, batch=512):
    model.eval()
    kriteria = nn.CrossEntropyLoss(reduction='sum')
    total_loss, benar = 0.0, 0
    for xb, yb in loader(ds, batch, False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total_loss += kriteria(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(ds), benar / len(ds)

jumlah_parameter = sum(p.numel() for p in buat_model().parameters())
print('jumlah parameter:', jumlah_parameter)
assert jumlah_parameter == 101_770, 'arsitektur belum sesuai protokol'

In [ ]:
def jalankan(nama_opt, lr, batch=BATCH, epoch=EPOCH, scheduler=None):
    """TODO 5: satu fungsi pelatihan untuk SELURUH run.

    Wajib mengembalikan (model, riwayat, catatan) dengan:
      riwayat : dict berisi list train_loss, val_loss, val_acc per epoch
      catatan : dict satu baris metrics.csv, memuat sekurangnya run_id, seed,
                optimizer, learning_rate, scheduler, batch_size, n_update,
                train_loss, val_loss, val_acc, grad_norm_mean, runtime_s
    Jangan lupa zero_grad -> backward -> step, dan catat norma gradien
    sebelum step.
    """
    raise NotImplementedError

## C. Baseline dan tiga learning rate - bagian dari 25 poin

Jalankan SGD pada $\eta \in \{0{,}001;\;0{,}1;\;1{,}0\}$, tampilkan ketiga kurva dalam satu grafik, lalu cocokkan dengan tabel gejala pada modul.

In [ ]:
# TODO 6: jalankan tiga learning rate, kumpulkan catatan dan riwayatnya,
#         lalu tampilkan satu grafik validation loss berlabel lengkap.
raise NotImplementedError

**Pembacaan kurva.** Cocokkan setiap kurva dengan gejala pada modul dan sebutkan pula nilai `grad_norm_mean`-nya:

- $\eta=0{,}001$: TODO
- $\eta=0{,}1$: TODO
- $\eta=1{,}0$: TODO

## D. Sembilan run terkendali - 25 poin

Setiap optimizer diuji pada tiga learning rate-nya sendiri. Model diinisialisasi ulang dari seed yang sama sebelum setiap run.

In [ ]:
grid = {'sgd': [0.01, 0.1, 0.5],
        'momentum': [0.01, 0.05, 0.1],
        'adam': [1e-4, 1e-3, 1e-2]}

# TODO 7: jalankan seluruh kombinasi, simpan ke daftar `hasil`
#         dan riwayat tiap run ke dict `riwayat_semua`.
hasil, riwayat_semua = [], {}
raise NotImplementedError

tabel = pd.DataFrame(hasil)
print(tabel[['run_id', 'optimizer', 'learning_rate', 'val_loss', 'val_acc',
             'runtime_s']].sort_values('val_loss').to_string(index=False))
assert len(tabel) == 9, 'harus ada sembilan run inti'

## E. Memilih model dan evaluasi akhir - 20 poin

Pilih learning rate terbaik tiap optimizer dari **validation loss**, bandingkan ketiga pemenang, lalu evaluasi data uji **satu kali** pada satu model final.

In [ ]:
# TODO 8: tentukan pemenang tiap optimizer, tampilkan tabel perbandingan
#         (val_loss, val_acc, runtime_s) dan satu grafik tiga kurva.
raise NotImplementedError

In [ ]:
# TODO 9: latih ulang konfigurasi final, lalu evaluasi ds_uji SATU KALI.
#         Tampilkan confusion matrix dan sebutkan dua kelas yang paling sering tertukar.
raise NotImplementedError

**Checkpoint menit ke-95.** Tunjukkan tabel sembilan run dan grafik tiga pemenang kepada asisten sebelum melanjutkan ke bagian F.

## F. Scheduler dan batch size - 15 poin

Enam run tambahan, seluruhnya memakai optimizer dan learning rate terbaik Anda.

In [ ]:
# TODO 10: tiga run scheduler (None, 'step', 'cosine') pada anggaran epoch yang sama,
#          lalu tiga run batch size (32, 128, 512). Laporkan kolom n_update.
tambahan = []
raise NotImplementedError

df_tambahan = pd.DataFrame(tambahan)
print(df_tambahan[['run_id', 'scheduler', 'batch_size', 'n_update',
                   'val_loss', 'val_acc', 'runtime_s']].to_string(index=False))
assert len(df_tambahan) == 6, 'harus ada enam run tambahan'

In [ ]:
# TODO 11: gabungkan seluruh run ke satu metrics.csv.
semua = pd.concat([tabel, df_tambahan], ignore_index=True)
semua.insert(0, 'module', 'M03')
semua.insert(1, 'student_id', NIM)
semua.to_csv(f'M03_{NIM}_metrics.csv', index=False)
print(f'{len(semua)} baris tersimpan')
assert len(semua) >= 15, 'metrics.csv minimal berisi 15 run'

## G. Pertanyaan analisis - bagian dari 15 poin

1. Apakah optimizer dengan validation loss terendah juga yang tercepat? TODO
2. Mengapa learning rate terbaik Adam jauh lebih kecil daripada SGD? TODO
3. Apa yang terjadi saat learning rate terlalu besar, dan pada epoch keberapa gejalanya terlihat? TODO
4. Apakah scheduler memperbaiki hasil pada anggaran epoch yang sama? Bila tidak, mengapa? TODO
5. Batch size mana yang Anda rekomendasikan bila anggaran diukur dalam **waktu**, bukan epoch? TODO

**Keterbatasan.** Modul ini memakai subset $12\,000$ citra dan $5$ epoch. Sebutkan apa yang tidak boleh disimpulkan dari hasil ini: TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, versi library, dan device tercantum.
- [ ] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [ ] Split, normalisasi, dan jumlah parameter lolos sel pemeriksaan.
- [ ] Seluruh run memanggil `jalankan()` yang sama.
- [ ] `metrics.csv` memuat minimal 15 run, termasuk yang gagal.
- [ ] Pemenang dipilih dari validation loss; data uji dipakai satu kali.
- [ ] Grafik berlabel lengkap dan keterbatasan subset disebutkan.
- [ ] Notebook lolos *Restart Kernel and Run All*.